# Phase 6 — Module 3 validation

Held-out datasets vs the two frozen rules. Predictions saved before training; scored after.


In [4]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
from virgo import frozen_rules as fr
from experiments import predict_module3 as pm, score_module3 as sm
fr.HELDOUT

['citeseer_linqs',
 'proteins',
 'pubmed',
 'actor',
 'minesweeper',
 'amazon_photo']

## 1 · Predict — before training

Both frozen rules are **link-prediction** rules (Module 2 found no node-classification rule).


In [5]:
pred, added = pm.freeze_predictions(fr.HELDOUT)
print(f"newly frozen: {added or 'none (already saved before training)'}")

# Display only: a disagreement is decided by the LEAD rule (frozen_rules.LEAD = rule 1), so one verdict, one rule named.
NAMES = {"rule1": "R1 (adj_h)", "rule2": "R2 (largest_comp_frac)"}
split = pred["rule1_pred"] != pred["rule2_pred"]
lead = pred[f"{fr.LEAD.name}_pred"] + f"  ({NAMES[fr.LEAD.name]})"
shown = pred.drop(columns=["tasks"]).assign(predicted_verdict=pred["predicted_verdict"].where(~split, lead))

display(
    shown.round(4)
    .rename(columns={
        "homophily_adjusted": "adj_h",
        "rule1_interval": "R1 (adj_h) interval",
        "rule1_pred": "R1 (adj_h) prediction",
        "largest_component_frac": "largest_comp_frac",
        "rule2_interval": "R2 (largest_comp_frac) interval",
        "rule2_pred": "R2 (largest_comp_frac) prediction",
        "predicted_verdict": "predicted LP verdict",
    })
    .style
    .format(na_rep="—")
    .hide(axis="index")
    .set_table_styles([
        {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed"), ("font-size", "11px")]},
        {"selector": "th", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
        {"selector": "td", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
    ])
)

newly frozen: none (already saved before training)


dataset,domain,adj_h,R1 (adj_h) interval,R1 (adj_h) prediction,largest_comp_frac,R2 (largest_comp_frac) interval,R2 (largest_comp_frac) prediction,predicted LP verdict
citeseer_linqs,citation,0.673100,"(0.0926, 0.3613)",keep original,0.646400,"(0.9177, 1.0)",keep original,keep original
proteins,biological,0.355200,"(0.0926, 0.3613)",keep original,0.014300,"(0.9177, 1.0)",keep original,keep original
pubmed,citation,0.686000,"(0.0926, 0.3613)",keep original,1.000000,"(0.9177, 1.0)",augment,keep original (R1 (adj_h))
actor,film,0.002800,"(0.0926, 0.3613)",augment,1.000000,"(0.9177, 1.0)",augment,augment
minesweeper,grid,0.009400,"(0.0926, 0.3613)",augment,1.000000,"(0.9177, 1.0)",augment,augment
amazon_photo,co-purchase,0.785000,"(0.0926, 0.3613)",keep original,0.978700,"(0.9177, 1.0)",augment,keep original (R1 (adj_h))


## 2 · Verdict vs actual — after training


In [6]:
scored = sm.score(fr.HELDOUT)
scored.to_csv(sm.SCORED_CSV, index=False)
for r in fr.FROZEN_RULES:
    col = list(scored[f"{r.name}_correct"])
    c = [v for v in col if isinstance(v, bool)]
    skipped = sorted({str(v) for v in col if not isinstance(v, bool)})
    print(f"{r.name} ({r.predictor} {r.op} {r.point}): "
          + (f"{sum(c)}/{len(c)} correct" if c else "nothing scored yet")
          + (f"  [not scored: {', '.join(skipped)}]" if skipped else ""))

# One-line scoreboard: how many held-out datasets, and how many each rule called right.
hits = {r.name: [v for v in scored[f"{r.name}_correct"] if isinstance(v, bool)] for r in fr.FROZEN_RULES}
print(f"STATS  datasets {len(scored)}  |  R1 (adj_h) {sum(hits['rule1'])}/{len(hits['rule1'])} correct  |  "
      f"R2 (largest_comp_frac) {sum(hits['rule2'])}/{len(hits['rule2'])} correct  |  "
      f"{len(scored) - len(hits['rule1'])} not scored (tie / pending)")

# Display only: same lead-rule tie-break as cell 1 - one verdict, one rule named.
NAMES = {"rule1": "R1 (adj_h)", "rule2": "R2 (largest_comp_frac)"}
split = scored["rule1_pred"] != scored["rule2_pred"]
lead = scored[f"{fr.LEAD.name}_pred"] + f"  ({NAMES[fr.LEAD.name]})"
shown = (scored.drop(columns=["rule1_correct", "rule2_correct"])   # per-rule accuracy printed above; the CSV keeps both
         .assign(predicted_verdict=scored["predicted_verdict"].where(~split, lead)))

display(
    shown.round(4)
    .rename(columns={
        "homophily_adjusted": "adj_h",
        "largest_component_frac": "largest_comp_frac",
        "rule1_pred": "R1 (adj_h) prediction",
        "rule2_pred": "R2 (largest_comp_frac) prediction",
        "predicted_verdict": "predicted LP verdict",
        "actual_verdict": "actual LP verdict",
        "best_augmented": "best aug",
        "best_variant": "best variant",
    })
    .style
    .format(na_rep="—")
    .hide(axis="index")
    .set_table_styles([
        {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed"), ("font-size", "11px")]},
        {"selector": "th", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
        {"selector": "td", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
    ])
)

rule1 (homophily_adjusted < 0.227): 3/4 correct  [not scored: no decision]
rule2 (largest_component_frac > 0.9588): 3/4 correct  [not scored: no decision]
STATS  datasets 6  |  R1 (adj_h) 3/4 correct  |  R2 (largest_comp_frac) 3/4 correct  |  2 not scored (tie / pending)


dataset,adj_h,largest_comp_frac,R1 (adj_h) prediction,R2 (largest_comp_frac) prediction,predicted LP verdict,actual LP verdict,original,best aug,best variant
citeseer_linqs,0.673100,0.646400,keep original,keep original,keep original,keep original,0.621800,0.543700,centrality
proteins,0.355200,0.014300,keep original,keep original,keep original,keep original,0.672000,0.583400,centrality
pubmed,0.686000,1.000000,keep original,augment,keep original (R1 (adj_h)),tie,0.639200,0.623800,hybrid
actor,0.002800,1.000000,augment,augment,augment,augment,0.595300,0.661700,degree
minesweeper,0.009400,1.000000,augment,augment,augment,keep original,0.709300,0.667100,hybrid
amazon_photo,0.785000,0.978700,keep original,augment,keep original (R1 (adj_h)),tie,0.785700,0.780300,hybrid
